# BRCA recurrence: fused patient graphs and GNN comparison

Primary label: `PFS_STATUS`, with `1:PROGRESSION` as the positive class and `0:CENSORED` as the negative class. Censored patients were disease-free only through the end of follow-up. Outcome columns are not graph features.

Each patient graph is the 500-gene alteration neighborhood joined to PrimeKG on Entrez id (panel PPI, pathways, drugs, and breast/mammary diseases). The MLP baseline uses the same clinical fields and alteration multi-hot without those edges.

Run `python scripts/train_and_evaluate_gnns.py` to refresh the tables and figures below. The modeling choices are written up in `docs/recurrence_gnn.md`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path('..')
OUT = ROOT / 'outputs' / 'gnn_benchmark'
summary = pd.read_csv(OUT / 'metrics_summary.csv')
folds = pd.read_csv(OUT / 'metrics_folds.csv')
audit = json.loads((OUT / 'label_audit.json').read_text())
print(
    f"PFS cohort: {audit['patients']} patients, {audit['positives']} progressions, "
    f"{audit['negatives']} censored. PrimeKG slice edges: {audit['primekg_edges']}."
)
print(audit['note'])
summary.round(3)

## Comparison plots

ROC and precision-recall curves use pooled out-of-fold probabilities. Bars are the mean and standard deviation across the five PFS folds. The confusion matrix uses the threshold chosen inside each training fold.

In [ ]:
from IPython.display import Image, Markdown, display

for name in ['roc_curves.png', 'pr_curves.png', 'comparison_bars.png', 'loss_curves.png', 'confusion_matrix.png']:
    display(Markdown(f'### {name}'))
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'summary.md').read_text()))

## Example paths

Paths are walks that exist in the fused graph: patient to sample to an altered gene, then a PrimeKG pathway, drug, or disease. Relation labels are the stored edge types. A drug is not described as causing resistance unless PrimeKG contains that edge.

In [ ]:
paths = pd.read_csv(OUT / 'explanations.csv')
paths.head(12)